In [20]:
from ngsolve import *
from netgen.geom2d import unit_square

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
fes = H1(mesh, order=1)
uh = GridFunction(fes)

u = CF((1, 2, 3, 4, 5, 6), dims=(2, 3))

* 代码生成了一个 2x3 的矩阵：

<div style="text-align: center;">
  <table style="margin: 0 auto;">
    <tr>
      <td>1</td>
      <td>2</td>
      <td>3</td>
    </tr>
    <tr>
      <td>4</td>
      <td>5</td>
      <td>6</td>
    </tr>
  </table>
</div>

总结原则：

`u = CF((1, 2, 3, 4, 5, 6), dims=(m, n))`，将元素排列成m行n列，逐行填充


**有两种语法**

* `u[1,0]`的语法中，索引的是第二行第一列的元素，4。如果是`u[2,0]`则超出行数
* `u[2][0]`的语法中，第一个括号索引的是列数（第三列），第二个括号索引的是行数（第一行）。这正好是`u[0,2]`

In [38]:
## 输出第二行
for ii in range(3):
    uh.Set(u[1,ii])
    print(uh.vec[0])

4.0
5.0
6.0


In [35]:
## 输出第一行
for ii in range(3):
    uh.Set(u[ii][0])
    print(uh.vec[0])

1.0
2.0
3.0


### CF矩阵和向量的乘法

用CF矩阵和向量实现： 

\begin{equation}
\begin{pmatrix}
1 & 2 & 3 \\
4 & 5 & 6
\end{pmatrix}
\begin{pmatrix}
1 \\
2 \\
3
\end{pmatrix}
=
\begin{pmatrix}
14 \\
32
\end{pmatrix}
\end{equation}


In [39]:
v = CF((1,2,3))
res = u*v

In [44]:
for ii in range(2):
    uh.Set(res[ii])
    print(uh.vec[ii])

14.0
32.0


### 向量与向量的内积

In [55]:
res = v*v
uh.Set(res)
print(uh.vec[0])

14.0


### 转置
* 转置是矩阵形式的CF特有的，向量不具备

In [56]:
v.trans

NgException: Transpose of non-matrix called

* 矩阵索引以及矩阵乘法（需要维度符合）

\begin{equation}
u \cdot u^T =
\begin{pmatrix}
1 & 2 & 3 \\
4 & 5 & 6
\end{pmatrix}
\cdot
\begin{pmatrix}
1 & 4 \\
2 & 5 \\
3 & 6
\end{pmatrix}
=
\begin{pmatrix}
14 & 32 \\
32 & 77
\end{pmatrix}
\end{equation}

\begin{equation}
\begin{pmatrix}
1 & 4 \\
2 & 5 \\
3 & 6
\end{pmatrix}
\cdot
(\begin{pmatrix}
2 & 3 \\
5 & 6
\end{pmatrix}
+\begin{pmatrix}
1 & 2\\
4 & 5
\end{pmatrix}
)
=
\begin{pmatrix} 39 & 49 \\ 51 & 65 \\ 63 & 81 \end{pmatrix}
\end{equation}

* u*u对于一般维度的矩阵是不可以的

In [89]:
res = u*(u.trans)
# print 第一行
r_list = []
for ii in range(2):
    uh.Set(res[0,ii])
    r_list.append(uh.vec[0])
print(r_list)


res = (u.trans)*(u[:2,:2]+u[:2,1:3])
# print 第一行
for jj in range(3):
    r_list = []
    for ii in range(2):
        uh.Set(res[jj,ii])
        r_list.append(uh.vec[0])
    print(r_list)

[14.0, 32.0]
[39.0, 49.0]
[51.0, 65.0]
[63.0, 80.99999999999999]


* 维度不匹配的矩阵乘法

In [57]:
res = u.trans*v
uh.Set(res)
print(uh.vec[0])

NgException: Matrix dimensions don't fit: mat is 3 x 2, vec is 3

* 向量矩阵乘法中，向量只能乘在右侧，不能乘在左侧

In [65]:
try:
    res = v*u.trans
    uh.Set(res)
    print(uh.vec[0])
except:
    res = v*u
    uh.Set(res)
    print(uh.vec[0])

NgException: T_MultVecVec : dimensions don't fit

* 可以将向量写成一个矩阵，一个维度是1；例如一行三列的矩阵是一个行向量。

In [96]:
w = CF((1,2,3), dims=(1,3))
res = w*u.trans
uh.Set(res[0])
print(uh.vec[0])

14.0
